# Aarohan-350M — Kaggle SFT Round 3 (Loss Mask Fixed) on TPU v5e-8

**Run AFTER pre-training is complete.**

Fine-tunes **Aarohan-350M** on ~146K instruction-response pairs using all 8 TPU chips
via `xmp.spawn` — the same multi-chip pattern used by the pre-training script.

**What was wrong in Rounds 1 & 2:**
- Round 1: Only 3 epochs, lr=1e-5 — too weak, model didn't converge
- Round 2: `tokenizer.decode()` silently strips `<|im_start|>` / `<|im_end|>` tokens,
  breaking character-position counting → **loss mask was all zeros → model learned NOTHING**

**Round 3 fix:**
- ✅ Loss mask now uses **token sub-sequence matching** (not character counting)
- ✅ Proven to work locally: `Mask sum: 7` on a test sample
- ✅ 6 epochs, lr=3e-5
- ✅ Starting from clean `pretrain_best.pt`

**Instructions:**
1. Enable TPU: Settings → Accelerator → **TPU v5e-8**
2. Attach your pre-training notebook output (which contains `best.pt`) as input data
3. Add Kaggle Secrets: `WANDB_API_KEY`, `GITHUB_TOKEN`
4. Click **Save Version** (Run in background — ~3h job)

**Estimated time:** ~3 hours on TPU v5e-8 (6 epochs ✅)

In [ ]:
# ── Cell 1: Verify TPU + fix TensorFlow conflict ─────────────
import torch
print(f'PyTorch: {torch.__version__}')

import subprocess
result = subprocess.run(['pip', 'uninstall', '-y', 'tensorflow'], capture_output=True, text=True)
if 'Successfully' in result.stdout:
    print('tensorflow uninstalled')
subprocess.run(['pip', 'install', 'tensorflow-cpu', '-q'], capture_output=True)
print('tensorflow-cpu installed')

import os
os.environ.pop('CLOUD_TPU_TASK_ID', None)
os.environ.pop('TPU_PROCESS_ADDRESSES', None)

import torch_xla.core.xla_model as xm
print(f'✅ torch_xla imported successfully (TPU ready for spawn)')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────
!pip install -q wandb tokenizers datasets pyyaml

In [ ]:
# ── Cell 3: Clone latest code (Round 3 fix is in training/sft.py) ─
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GITHUB_REPO = 'Abhik2005/se-llm-data'

if not os.path.exists('/kaggle/working/se-llm-350m'):
    try:
        token    = secrets.get_secret('GITHUB_TOKEN')
        repo_url = f'https://{token}@github.com/{GITHUB_REPO}.git'
    except Exception:
        repo_url = f'https://github.com/{GITHUB_REPO}.git'
    !git clone {repo_url} /kaggle/working/se-llm-350m
else:
    # Pull latest — IMPORTANT: gets the Round 3 loss mask fix
    !git -C /kaggle/working/se-llm-350m pull

%cd /kaggle/working/se-llm-350m

# Verify the fix is present
!grep -n '_build_loss_mask' training/sft.py
print('\n✅ If you see _build_loss_mask (not _build_loss_mask_by_text), the fix is active!')

In [ ]:
# ── Cell 4: Generate SFT instruction dataset ──────────────────
import os

os.makedirs('data/sft', exist_ok=True)

if not os.path.exists('data/sft/sft_data.jsonl'):
    print('Generating SFT dataset from HuggingFace...')
    !python data/sft_data.py
else:
    with open('data/sft/sft_data.jsonl') as f:
        n = sum(1 for _ in f)
    size_mb = os.path.getsize('data/sft/sft_data.jsonl') / 1e6
    print(f'✅ SFT dataset ready: {n:,} samples | {size_mb:.1f} MB')

In [ ]:
# ── Cell 5: Link tokenizer ────────────────────────────────────
import os

os.makedirs('tokenizer', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('checkpoints_sft', exist_ok=True)

tok_candidates = [
    '/kaggle/input/datasets/vedase/se-llm-data/tokenizer.json',
    '/kaggle/input/se-llm-data/tokenizer.json',
]
for tok_src in tok_candidates:
    if os.path.exists(tok_src):
        tok_dest = 'tokenizer/tokenizer.json'
        if not os.path.exists(tok_dest):
            os.symlink(tok_src, tok_dest)
        print(f'✅ Linked tokenizer from {tok_src}')
        break
else:
    print('WARNING: tokenizer.json not found — add the se-llm-data dataset in the sidebar!')

# Quick sanity check on loss mask
from tokenizers import Tokenizer
tok = Tokenizer.from_file('tokenizer/tokenizer.json')
test_ids = tok.encode('<|im_start|>system\nYou are Aarohan.<|im_end|>\n<|im_start|>user\ntest<|im_end|>\n<|im_start|>assistant\nHello!<|im_end|>\n').ids
im_end_id   = tok.token_to_id('<|im_end|>')
asst_prefix = tok.encode('<|im_start|>assistant\n').ids
prefix_len  = len(asst_prefix)
mask = [0] * len(test_ids)
in_assistant = False
i = 0
while i < len(test_ids):
    if not in_assistant:
        if i + prefix_len <= len(test_ids) and test_ids[i:i+prefix_len] == asst_prefix:
            in_assistant = True; i += prefix_len; continue
        i += 1
    else:
        mask[i] = 1
        if test_ids[i] == im_end_id: in_assistant = False
        i += 1
if sum(mask) > 0:
    print(f'✅ Loss mask sanity check PASSED! Mask sum = {sum(mask)} (tokens: {[tok.decode([test_ids[i]]) for i,m in enumerate(mask) if m]})')
else:
    print('❌ LOSS MASK IS BROKEN — DO NOT PROCEED!')

In [ ]:
# ── Cell 6: Load pre-trained Aarohan-350M checkpoint ──────────
# Attach the OUTPUT of your pre-training notebook (has best.pt) as input data.
import os, shutil, glob, torch

search_paths = glob.glob('/kaggle/input/**/best.pt', recursive=True)

base_checkpoint = None

if search_paths:
    src  = search_paths[0]
    dest = 'checkpoints/pretrain_best.pt'
    if not os.path.exists(dest):
        print(f'Copying {src} → {dest}  (2.36 GB, please wait...)')
        shutil.copy2(src, dest)
    base_checkpoint = dest

    ckpt = torch.load(dest, map_location='cpu', weights_only=False)
    print(f'✅ Pre-trained checkpoint loaded:')
    print(f'   Model:     {ckpt.get("model_config", {}).get("name", "unknown")}')
    print(f'   Step:      {ckpt.get("step", 0):,}')
    print(f'   Tokens:    {ckpt.get("tokens_processed", 0)/1e9:.3f}B')
    print(f'   Val Loss:  {ckpt.get("val_loss", 0):.4f}')
else:
    print('❌ No best.pt found!')
    print('   → In the Kaggle sidebar, click + Add Data → Notebooks')
    print('   → Find your pre-training notebook and add its output')

In [ ]:
# ── Cell 7: Login to W&B ──────────────────────────────────────
import wandb
from kaggle_secrets import UserSecretsClient

try:
    secrets   = UserSecretsClient()
    wandb_key = secrets.get_secret('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print('✅ W&B logged in')
except Exception as e:
    print(f'W&B login skipped: {e}')

In [ ]:
# ── Cell 8: RUN INSTRUCTION FINE-TUNING ON TPU ────────────────
# Round 3: Loss mask FIXED (token sub-sequence matching).
# Starting from pretrain_best.pt — clean base model.
# 6 epochs, lr=3e-5 — Expected: ~3 hours, loss should drop to ~1.6-2.0

assert base_checkpoint is not None, \
    'No checkpoint found! Attach pre-training notebook output in sidebar.'

cmd = f'python training/sft.py --config configs/350m.yaml --base-checkpoint {base_checkpoint}'
print(f'Running: {cmd}\n')
!{cmd}

In [ ]:
# ── Cell 9: Quick quality test ─────────────────────────────────
import torch, os, sys, glob
sys.path.insert(0, '/kaggle/working/se-llm-350m')

from evaluation.generate import load_model_from_checkpoint, load_tokenizer, chat_turn

device = torch.device('cpu')

ckpt_path = 'checkpoints_sft/sft_final.pt'
if not os.path.exists(ckpt_path):
    pts = sorted(glob.glob('checkpoints_sft/*.pt'))
    ckpt_path = pts[-1] if pts else None

if ckpt_path and os.path.exists(ckpt_path):
    model, cfg = load_model_from_checkpoint(ckpt_path, device)
    tokenizer  = load_tokenizer('tokenizer/tokenizer.json')

    test_prompts = [
        'Write a JavaScript function that sums two numbers.',
        'What is the difference between a stack and a queue?',
        'Write a Python function to check if a number is prime.',
    ]

    for prompt in test_prompts:
        print(f'\n{"─"*55}')
        print(f'User: {prompt}')
        response = chat_turn(model, tokenizer, prompt, device=device, max_new_tokens=200)
        print(f'Aarohan:\n{response}')

    print('\n✅ Aarohan-350M SFT Round 3 quality check complete!')
else:
    print('No SFT checkpoint found — check if training completed successfully.')

In [ ]:
# ── Cell 10: Done! ────────────────────────────────────────────
import glob, os

print('SFT checkpoints in checkpoints_sft/')
for ckpt in sorted(glob.glob('checkpoints_sft/*.pt')):
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f'  {os.path.basename(ckpt):40s}  {size_mb:.0f} MB')

print()
print('✅ Aarohan-350M SFT Round 3 complete!')
print('   Download checkpoints_sft/sft_final.pt and test locally:')
print('   python evaluation/generate.py \\')
print('     --checkpoint checkpoints/sft_final.pt \\')
print('     --mode chat')